In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from langchain_openai import ChatOpenAI
from agents.sql_agent import *
from agents.orchestrator import *

In [ ]:
max_sql_retries = 3
chat_history = 5
api_key = os.getenv("MY_API_KEY")

In [ ]:
db_name = "finance.db"

db_description = """
Tables:

1. outcome
Columns:
- Date (DATE): expense date in YYYY-MM-DD format. There may be multiple expenses on the same day
- Value (FLOAT): amount of money spent (in polish zloty)
- Category (TEXT): type of expense. Options: czynsz, trip, services, yummy, restraunt, food, present, clothes, transport, outdoors, holiday, work, marta, projeb, health, electronics, house, cosmetics, education, charity, unknown, remont, jewellery
- Was planned (TEXT): whether the outcome was planned ('yes', 'no')
- Description (TEXT): a subcategory of category

2. income
Columns:
- Date (DATE): same as above
- Value (FLOAT): amount of money received
- Category (TEXT): type of income. Options: same as for outcome + salary, stypendium
- Constant (TEXT): whether the income is constant ('yes', 'no')
- Description (TEXT): same as above
"""

In [ ]:
model = ChatOpenAI(model="gpt-4", api_key=api_key)
sql_agent = SQLAgent(model, db_name, db_description, max_sql_retries)
buddy = DataBuddyAgent(model, sql_agent, chat_history)

In [ ]:
user_input = "I want all expenses from zabka from 2025"
user_input = "was i spending more than i year in 2025?"
user_input = "ile miesiecnie srednio trace na jedzenie"
user_input = "сколько денег уходит в среднем месяц на косметику c 2026 года"
# user_input = "What's the weather in London now?"
# user_input = "i want to see how many i spend in this year so far on clothes"

In [ ]:
buddy = DataBuddyAgent(model, sql_agent, chat_history)
buddy.chat("сколько денег уходит в среднем месяц на косметику c 2026 года")

In [ ]:
buddy.chat("не, сколько в месяц я трачу на косметику с 2026 года?")

In [ ]:
buddy.chat("и теперь с последнего сделай среднюю")

In [ ]:
a

In [ ]:
while True:
    user_input = input("Your question (or 'quit'): ")
    if user_input.lower() == "quit":
        break
    res = buddy.chat(user_input)
    print("\nAnswer:\n", res["answer"])
    print("Query:\n", res["last query"])
    print("Intent:\n", res["intent"])
    print("Tokens spent:\n", res["token spent"])

In [ ]:
def query_executor(query, db="finance.db"):
    """Executes SQL query and returns a DataFrame. Returns error if invalid."""
    try:
        with sqlite3.connect(db) as conn:
            df = pd.read_sql_query(query, conn)
        return df, None
    except Exception as e:
        return None, str(e)

In [ ]:
query_executor("SELECT * FROM outcome WHERE Category = 'clothes' AND Date LIKE '2025-01%'")